In [1]:
import yaml
import json
from app.datasets.loader import load_multiple_test_cases
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests
from app.utils.db import save_results_on_cosmos

In [2]:
file_list = [
  './app/data/raw/tramites.xlsx',
  './app/data/raw/accesibilidad.xlsx',
  './app/data/raw/descubrir.xlsx',
  './app/data/raw/solicitudes.xlsx',
  './app/data/raw/organigrama.xlsx'
]

test_config = {
  'TIMINGS': {'test': False, 'report': False},
  'TOKENS': {'test': False, 'report': False},
  'FOUNDRYS': {'test': False, 'report': False},
  'TRIAGE': {'test': False, 'report': False},
  'ROUTER': {'test': False, 'report': False},
  'GROUNDING': {'test': False, 'report': False},
  'REFORMULATE': {'test': False, 'report': False},
  'SAVE_RESULTS': False,
  'PATH': './app/data/processed/reports/report'
}   

df = load_multiple_test_cases(file_list)
df = validate_dataset_schema(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)

In [ ]:
# responses = client.query_batch(df['user_input'],df['reference'])

In [ ]:
# save_responses_in_json, response_file_path = client.save_api_responses(responses)
response_file_path = './app/data/processed/outcome_20260205-120729.json'

In [ ]:
with open(response_file_path, 'r', encoding='UTF-8') as f:
  data = json.load(f)

results = run_tests(
  config = test_config, 
  data = data, 
  df = df, 
  timestamp = str(response_file_path)
)

In [54]:
import json
import time # usar time.perf_counter()
import requests
import pandas as pd
from pathlib import Path
from app.datasets.loader import load_test_cases
reformulate_dataset = load_test_cases('./app/data/raw/reformulate.xlsx')

In [61]:
def send_reformulate_requests(config: dict, dataset: pd.DataFrame) -> list[dict]:
  all_responses = []
  headers = config.get('headers', {'Content-Type': 'application/json'})
  url = config['agent'].get('base_url')
  
  for _, row in dataset.iterrows():
    body = {
    'id': '',
    }
    questions = row['Pregunta']
    
    if '\n' in questions:
      questions = questions.split('\n')
      
      for question in questions:
        body['question'] = question
        
        try:
          response = requests.post(url=url, headers=headers, json=body)
          data = response.json()
          
          if body['id'] == '':
            body['id'] = data.get('id')
            
          all_responses.append(data)      
          
        except Exception as e:
          print(e)
      
  return all_responses

def save_reformualte_responses(responses: list[dict], output_path: str = './app/data/processed/reformulate_outcome.json', pretty_print: bool = True): 
    output_path = Path(output_path)
    output_path.parent.mkdir(parents= True, exist_ok=True)

    timestr = time.strftime("%Y%m%d-%H%M%S")
    base_name = output_path.stem
    extension = output_path.suffix
    timestamped_path = output_path.parent / f"{base_name}_{timestr}{extension}"
    
    output_data = [item for item in responses]
    try:
      with open(timestamped_path, 'w', encoding='UTF-8') as f:
        if pretty_print:
          f.write(json.dumps(output_data, indent=2,   ensure_ascii=False))
          return str(timestamped_path), timestamped_path

    except Exception as e:
      print('Failed to save responses to JSON')
      raise

In [48]:
reformulate_responses = send_reformulate_requests(config_data, reformulate_dataset)

In [ ]:
save_reformulate_responses, timestamped_path = save_reformualte_responses(reformulate_responses)

('app\\data\\processed\\reformulate_outcome_20260303-121549.json', WindowsPath('app/data/processed/reformulate_outcome_20260303-121549.json'))
